# Ch02-03: 랜덤 워크와 브라운 운동 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

Donsker 정리를 다양한 n에서 시각화합니다.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(42)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, n in zip(axes.flat, [10, 100, 1000, 10000]):
    t_grid = np.linspace(0, 1, 1000)
    for _ in range(5):
        steps = np.random.choice([-1, 1], n)
        S = np.cumsum(steps)
        S_normalized = np.interp(t_grid, np.linspace(0, 1, n), S / np.sqrt(n))
        ax.plot(t_grid, S_normalized, alpha=0.5)
    ax.set_title(f'n = {n}')
    ax.set_ylim(-3, 3)
plt.suptitle('Donsker 정리: S[nt]/√n → W(t)')
plt.tight_layout()
plt.show()

**결과 해석**: n이 증가할수록 경로가 점점 더 부드러운 브라운 운동에 가까워집니다. 이는 CLT의 함수적 확장입니다.

---
## 문제 2 풀이

GBM 경로를 생성하고 로그정규성을 검증합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

S0, mu, sigma, T = 100, 0.08, 0.2, 1.0
n_paths, n_steps = 10000, 252
dt = T / n_steps
np.random.seed(42)

paths = np.zeros((n_paths, n_steps + 1))
paths[:, 0] = S0
for t in range(n_steps):
    Z = np.random.standard_normal(n_paths)
    paths[:, t+1] = paths[:, t] * np.exp((mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z)

final_prices = paths[:, -1]
log_returns = np.log(final_prices / S0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i in range(min(100, n_paths)):
    axes[0].plot(paths[i], alpha=0.1, color='blue')
axes[0].set_title('GBM 경로')

axes[1].hist(log_returns, bins=50, density=True, alpha=0.7, label='시뮬레이션')
x = np.linspace(log_returns.min(), log_returns.max(), 100)
theory_mean = (mu - 0.5*sigma**2) * T
theory_std = sigma * np.sqrt(T)
axes[1].plot(x, stats.norm.pdf(x, theory_mean, theory_std), 'r-', linewidth=2, label='이론')
axes[1].set_title('로그 수익률 분포')
axes[1].legend()

stats.probplot(log_returns, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot')
plt.tight_layout()
plt.show()
print(f"이론: mean={theory_mean:.4f}, std={theory_std:.4f}")
print(f"실험: mean={log_returns.mean():.4f}, std={log_returns.std():.4f}")

**결과 해석**: GBM의 로그 수익률은 정규분포를 따르므로 최종 가격은 로그정규분포입니다. 실제 주가는 두꺼운 꼬리와 변동성 클러스터링 때문에 GBM과 차이가 있습니다.

---
## 문제 3 풀이

2D vs 3D 랜덤 워크의 재귀성을 비교합니다.

In [ ]:
import numpy as np
np.random.seed(42)
n_walks = 5000
max_steps = 100000

returns_2d, returns_3d = 0, 0
for _ in range(n_walks):
    # 2D
    pos = np.array([0, 0])
    for step in range(1, max_steps):
        move = np.random.choice(4)
        pos += [(1,0),(-1,0),(0,1),(0,-1)][move]
        if np.all(pos == 0):
            returns_2d += 1
            break

    # 3D
    pos = np.array([0, 0, 0])
    for step in range(1, max_steps):
        move = np.random.choice(6)
        pos += [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)][move]
        if np.all(pos == 0):
            returns_3d += 1
            break

print(f"2D 복귀 확률: {returns_2d/n_walks:.4f} (이론: 1.0)")
print(f"3D 복귀 확률: {returns_3d/n_walks:.4f} (이론: ~0.3405)")

**결과 해석**: Pólya의 정리: 2D 대칭 랜덤 워크는 확률 1로 원점에 복귀하지만(재귀적), 3D에서 복귀 확률은 약 0.3405입니다(일시적). 이는 고차원에서 '공간이 넓어서' 원점을 놓치기 때문입니다.

---
## 확장 토론

브라운 운동은 확률론, 편미분방정식, 금융수학의 핵심입니다. Itô 적분과 확률 미적분학은 GBM을 일반화한 확률 미분 방정식(SDE)의 기초입니다.